In [1]:
# =====================================================
# rule_engine.py - Standalone Rule Engine
# Evaluates IF-THEN rules from Knowledge Graph
# =====================================================

import json
import re
from typing import Dict, List, Any
from datetime import datetime
import os
import shutil


class SurvivalRuleEngine:
    """
    Rule Engine that evaluates IF-THEN rules from Knowledge Graph
    """

    def __init__(self, knowledge_graph_data: Dict = None):
        """Initialize Rule Engine with Knowledge Graph data"""
        self.graph_data = knowledge_graph_data
        self.rules = []
        self.context = {}

        if self.graph_data:
            self._compile_rules()
            print(f"[RuleEngine] Initialized with {len(self.rules)} rules")
        else:
            print("[RuleEngine] Initialized with no data. Load KG first.")

    def load_from_json(self, filename="knowledge_graph.json"):
        """Load Knowledge Graph from JSON file"""
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                self.graph_data = json.load(f)
            self._compile_rules()
            print(f"[RuleEngine] Loaded KG from {filename} with {len(self.rules)} rules")
            return True
        except FileNotFoundError:
            print(f"[RuleEngine] File not found: {filename}")
            return False

    def _compile_rules(self):
        """Compile rules from Knowledge Graph"""
        nodes = self.graph_data.get("nodes", [])
        edges = self.graph_data.get("edges", [])
        self.rules = []

        for node in nodes:
            if node.get("type") == "condition":
                # Find skill node connected to this condition
                skill_id = None
                for edge in edges:
                    if edge.get("from") == node["id"] and edge.get("relation") == "leads_to":
                        skill_id = edge.get("to")
                        break

                if skill_id:
                    # Find conclusion
                    conclusion_text = None
                    for edge in edges:
                        if edge.get("from") == skill_id and edge.get("relation") == "results_in":
                            conc_node = next((n for n in nodes if n["id"] == edge.get("to")), None)
                            if conc_node:
                                conclusion_text = conc_node.get("text")
                            break

                    if conclusion_text:
                        skill_node = next((n for n in nodes if n["id"] == skill_id), {})
                        self.rules.append({
                            "condition": node["text"],
                            "conclusion": conclusion_text,
                            "skill": skill_node.get("name", "Unknown"),
                            "confidence": skill_node.get("confidence", 1.0),
                            "condition_id": node["id"],
                            "skill_id": skill_id
                        })

        # Sort by confidence
        self.rules.sort(key=lambda x: x.get("confidence", 0), reverse=True)

    def evaluate(self, input_data: Dict[str, Any]) -> List[Dict]:
        """Evaluate rules against input data"""
        results = []
        self.context = input_data.copy()

        for rule in self.rules:
            if self._check_condition(rule["condition"], input_data):
                results.append({
                    "rule": rule,
                    "result": self._evaluate_conclusion(rule["conclusion"], input_data),
                    "status": "triggered",
                    "confidence": rule["confidence"],
                    "skill": rule["skill"]
                })
            else:
                results.append({
                    "rule": rule,
                    "status": "not_triggered"
                })

        return results

    def _check_condition(self, condition: str, data: Dict) -> bool:
        """Check if condition is satisfied"""
        try:
            # Handle boolean values
            for key, value in data.items():
                if key in condition:
                    if isinstance(value, bool):
                        if f"{key} = True" in condition or f"{key} == True" in condition:
                            return value is True
                        elif f"{key} = False" in condition or f"{key} == False" in condition:
                            return value is False
                    elif isinstance(value, (int, float)):
                        # Numeric comparisons
                        pattern = f"{key}\\s*([><=!]+)\\s*(\\d+)"
                        match = re.search(pattern, condition)
                        if match:
                            operator = match.group(1)
                            threshold = float(match.group(2))

                            if operator == ">":
                                return value > threshold
                            elif operator == "<":
                                return value < threshold
                            elif operator == ">=":
                                return value >= threshold
                            elif operator == "<=":
                                return value <= threshold
                            elif operator == "==":
                                return value == threshold
                    elif isinstance(value, str):
                        if f"{key} = '{value}'" in condition or f"{key} == '{value}'" in condition:
                            return True
            return False
        except Exception as e:
            return False

    def _evaluate_conclusion(self, conclusion: str, data: Dict) -> str:
        """Evaluate conclusion with actual values"""
        result = conclusion
        for key, value in data.items():
            if str(value) in result:
                result = result.replace(str(value), f"[{key}: {value}]")
        return result

    def get_rules_count(self) -> int:
        return len(self.rules)

    def get_triggered_rules(self, input_data: Dict) -> List[Dict]:
        """Return only triggered rules"""
        results = self.evaluate(input_data)
        return [r for r in results if r.get("status") == "triggered"]

    def export_rules(self, filename="rules.json"):
        """Export compiled rules to JSON"""
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(self.rules, f, indent=2, ensure_ascii=False)
        print(f"[RuleEngine] {len(self.rules)} rules exported to {filename}")
        return self.rules

    def display_summary(self):
        """Display rules summary"""
        print("\n" + "="*70)
        print("RULE ENGINE SUMMARY")
        print("="*70)
        print(f"\nTotal Rules: {len(self.rules)}")

        if self.rules:
            print("\nSample Rules (First 5):")
            for i, rule in enumerate(self.rules[:5]):
                print(f"  {i+1}. IF {rule['condition']} THEN {rule['conclusion']}")
                print(f"     Skill: {rule['skill']} (Confidence: {rule['confidence']})")

        print("="*70)


# =====================================================
# RUN THE CODE
# =====================================================

if __name__ == "__main__":

    print("\n" + "="*70)
    print("SURVIVALVR-BD RULE ENGINE")
    print("Evaluates IF-THEN rules from Knowledge Graph")
    print("="*70 + "\n")

    # Mount Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted successfully!")
        DRIVE_PATH = "/content/drive/MyDrive/"
    except:
        print("Not in Colab. Saving locally.")
        DRIVE_PATH = "./"

    # Create output folder
    FOLDER_NAME = "SurvivalVR_Outputs"
    full_folder_path = os.path.join(DRIVE_PATH, FOLDER_NAME)
    os.makedirs(full_folder_path, exist_ok=True)
    print(f"Output folder: {full_folder_path}\n")

    # Load Knowledge Graph
    engine = SurvivalRuleEngine()

    # Try to load KG from multiple locations
    kg_loaded = False
    for kg_path in ["knowledge_graph.json",
                    os.path.join(full_folder_path, "knowledge_graph.json"),
                    "/content/knowledge_graph.json"]:
        if engine.load_from_json(kg_path):
            kg_loaded = True
            break

    if not kg_loaded:
        print("Creating sample rules...")
        engine.rules = [
            {
                "condition": "wind_speed > 60 AND pressure_drop > 20",
                "conclusion": "cyclone_approaching = True",
                "skill": "Cyclone Warning Recognition",
                "confidence": 0.90
            },
            {
                "condition": "cloud_type = 'cumulonimbus'",
                "conclusion": "storm_incoming = True",
                "skill": "Cloud Reading for Weather Prediction",
                "confidence": 0.88
            },
            {
                "condition": "birds_diving = True AND birds_count > 10",
                "conclusion": "fish_school_nearby = True",
                "skill": "Bird Behavior Reading",
                "confidence": 0.88
            },
            {
                "condition": "fish_oil_available = True AND dry_wood_available = True",
                "conclusion": "fire_possible = True",
                "skill": "Fish Oil Fire Making",
                "confidence": 0.92
            },
            {
                "condition": "water_depth > 10 AND seabed_type = 'mud'",
                "conclusion": "anchoring_safe = True",
                "skill": "Safe Anchoring Positions",
                "confidence": 0.87
            }
        ]

    engine.display_summary()

    # Test with sample input
    print("\n" + "="*70)
    print("TESTING WITH SAMPLE INPUT")
    print("="*70)

    test_input = {
        "wind_speed": 70,
        "pressure_drop": 25,
        "cloud_type": "cumulonimbus",
        "birds_diving": True,
        "birds_count": 15,
        "fish_oil_available": True,
        "dry_wood_available": True
    }

    print(f"\nInput Data: {json.dumps(test_input, indent=2)}")

    results = engine.evaluate(test_input)
    triggered = [r for r in results if r.get("status") == "triggered"]

    print(f"\nTriggered Rules: {len(triggered)} out of {len(results)}")
    for r in triggered:
        print(f"  - {r['skill']}: {r['result']} (Confidence: {r['confidence']})")

    # Export
    engine.export_rules("rules.json")

    # Copy to Drive
    try:
        shutil.copy("rules.json", full_folder_path)
        print(f"\nRules exported to: {full_folder_path}")
    except:
        pass

    print("\n" + "="*70)
    print("RULE ENGINE COMPLETE!")
    print("="*70)


SURVIVALVR-BD RULE ENGINE
Evaluates IF-THEN rules from Knowledge Graph

Not in Colab. Saving locally.
Output folder: ./SurvivalVR_Outputs

[RuleEngine] Initialized with no data. Load KG first.
[RuleEngine] File not found: knowledge_graph.json
[RuleEngine] File not found: ./SurvivalVR_Outputs/knowledge_graph.json
[RuleEngine] File not found: /content/knowledge_graph.json
Creating sample rules...

RULE ENGINE SUMMARY

Total Rules: 5

Sample Rules (First 5):
  1. IF wind_speed > 60 AND pressure_drop > 20 THEN cyclone_approaching = True
     Skill: Cyclone Warning Recognition (Confidence: 0.9)
  2. IF cloud_type = 'cumulonimbus' THEN storm_incoming = True
     Skill: Cloud Reading for Weather Prediction (Confidence: 0.88)
  3. IF birds_diving = True AND birds_count > 10 THEN fish_school_nearby = True
     Skill: Bird Behavior Reading (Confidence: 0.88)
  4. IF fish_oil_available = True AND dry_wood_available = True THEN fire_possible = True
     Skill: Fish Oil Fire Making (Confidence: 0.